#Domain-Specific Q&A Bot

By Akshat Kesharwani | Free tools only (Google Colab + free-tier APIs)

## Problem Statement

Support teams answer a lot of repeat questions (order status, refund policy, product specs). Companies like DoorDash use a fine-tuned Q&A model to pull the exact answer straight out of a document instead of a human reading through it every time. Here we fine-tune DistilBERT (a smaller, faster version of BERT) to do the same thing -- give it a question and a paragraph, it finds the answer span inside the paragraph.

### SECTION 1: Install & Import Libraries

In [2]:
!pip install -q transformers datasets accelerate evaluate openai

import os

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForQuestionAnswering,
    TrainingArguments, Trainer, DefaultDataCollator
)

# ---- Small AI helper: call Groq (free) ----
# Only using Groq here -- it's free, fast, and just needs an API key,
# no extra setup like Ollama needs.
# Get a free key at https://console.groq.com/keys
# (Update: as of June 2026 Groq retired the old Llama-3.3-70B model,
# the current one to use is "openai/gpt-oss-120b".)
from getpass import getpass
from openai import OpenAI

groq_key = getpass("Enter your Groq API key: ")
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_key)

def call_llm(prompt, max_tokens=300):
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
Enter your Groq API key: ··········


### SECTION 2: Load Domain Q&A Dataset (with robust fallback)

In [3]:
# heads up: HuggingFace changed their `datasets` library and stopped
# supporting the old "loading script" style datasets (squad is one of
# them). so first we try the normal way, then a parquet mirror, and if
# both of those fail we just make up a small dataset ourselves so the
# rest of the notebook still runs fine.
def robust_load_dataset(repo_candidates, split="train"):
    for repo_id in repo_candidates:
        try:
            print(f"[dataset] Trying load_dataset('{repo_id}', split='{split}')...")
            ds = load_dataset(repo_id, split=split)
            print(f"[dataset] Loaded '{repo_id}' successfully ({len(ds)} rows).")
            return ds
        except Exception as e:
            print(f"[dataset] '{repo_id}' standard load failed: {type(e).__name__}: {e}")
        try:
            parquet_path = f"hf://datasets/{repo_id}@refs/convert/parquet/default/{split}/*.parquet"
            print(f"[dataset] Trying Parquet fallback: {parquet_path}")
            ds = load_dataset("parquet", data_files=parquet_path, split="train")
            print(f"[dataset] Parquet fallback for '{repo_id}' succeeded ({len(ds)} rows).")
            return ds
        except Exception as e:
            print(f"[dataset] Parquet fallback for '{repo_id}' failed: {type(e).__name__}: {e}")
    print("[dataset] All remote candidates failed -- using built-in synthetic fallback data.")
    return None

raw_train = robust_load_dataset(["rajpurkar/squad", "squad", "squad_v2"], split="train")
raw_val = robust_load_dataset(["rajpurkar/squad", "squad", "squad_v2"], split="validation")

if raw_train is None or raw_val is None:
    # ---- Synthetic offline fallback -- guarantees the notebook always runs ----
    SYNTHETIC_QA = [
        {"context": "Our return policy allows customers to return any item within 30 days of purchase for a full refund, provided the item is unused and in its original packaging.",
         "question": "How many days do customers have to return an item?",
         "answers": {"text": ["30 days"], "answer_start": [61]}},
        {"context": "The company headquarters is located in Bengaluru, India, and employs over 2,000 people across engineering, sales, and support.",
         "question": "Where is the company headquarters located?",
         "answers": {"text": ["Bengaluru, India"], "answer_start": [39]}},
        {"context": "Employees are entitled to 18 days of paid annual leave, plus an additional 10 public holidays observed by the company each year.",
         "question": "How many days of paid annual leave do employees get?",
         "answers": {"text": ["18 days"], "answer_start": [26]}},
        {"context": "The premium subscription plan costs $29.99 per month and includes priority support, unlimited storage, and advanced analytics.",
         "question": "How much does the premium subscription cost?",
         "answers": {"text": ["$29.99 per month"], "answer_start": [30]}},
        {"context": "Server maintenance is scheduled every Sunday between 2 AM and 4 AM UTC, during which the API may be briefly unavailable.",
         "question": "When is server maintenance scheduled?",
         "answers": {"text": ["Sunday between 2 AM and 4 AM UTC"], "answer_start": [38]}},
        {"context": "New employees must complete a mandatory security training within their first 14 days of joining the company.",
         "question": "How long do new employees have to complete security training?",
         "answers": {"text": ["14 days"], "answer_start": [78]}},
        {"context": "The warranty on all electronics covers manufacturing defects for a period of 2 years from the date of purchase.",
         "question": "How long is the warranty period?",
         "answers": {"text": ["2 years"], "answer_start": [75]}},
        {"context": "Refund requests are typically processed within 5-7 business days and credited back to the original payment method.",
         "question": "How long does it take to process a refund?",
         "answers": {"text": ["5-7 business days"], "answer_start": [46]}},
        {"context": "The customer support team is available 24 hours a day, 7 days a week, via chat, email, and phone.",
         "question": "What hours is customer support available?",
         "answers": {"text": ["24 hours a day, 7 days a week"], "answer_start": [40]}},
        {"context": "Password reset links expire after 30 minutes for security reasons, after which a new link must be requested.",
         "question": "How long do password reset links stay valid?",
         "answers": {"text": ["30 minutes"], "answer_start": [36]}},
    ]
    expanded = SYNTHETIC_QA * 15  # replicate so the Trainer has a workable number of steps
    raw_train = Dataset.from_list(expanded[:120])
    raw_val = Dataset.from_list(expanded[120:135])
    print("[dataset] Using synthetic fallback: swap in a real Kaggle/HF dataset before reporting final metrics.")

train_data = raw_train.select(range(min(10000, len(raw_train))))  # 10K+ Q&A pairs when using the real dataset
val_data = raw_val.select(range(min(1000, len(raw_val))))

print("Train size:", len(train_data))
print("Sample:", train_data[0])

[dataset] Trying load_dataset('rajpurkar/squad', split='train')...


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

[dataset] Loaded 'rajpurkar/squad' successfully (87599 rows).
[dataset] Trying load_dataset('rajpurkar/squad', split='validation')...
[dataset] Loaded 'rajpurkar/squad' successfully (10570 rows).
Train size: 10000
Sample: {'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mar

### SECTION 3: Tokenize & Align Answer Positions

In [4]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions, examples["context"],
        max_length=384, truncation="only_second",
        return_offsets_mapping=True, padding="max_length"
    )
    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions, end_positions = [], []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # find context boundaries
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

tokenized_train = train_data.map(preprocess_function, batched=True, remove_columns=train_data.column_names)
tokenized_val = val_data.map(preprocess_function, batched=True, remove_columns=val_data.column_names)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

### SECTION 4: Load Pretrained Model

In [5]:
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
data_collator = DefaultDataCollator()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### SECTION 5: Fine-Tune with HuggingFace Trainer

In [6]:
use_fp16 = torch.cuda.is_available()  # fp16 only works with a real GPU -- Runtime > Change runtime type > T4 GPU
print("GPU available:", use_fp16)

training_args = TrainingArguments(
    output_dir="./qa-bot-checkpoints",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,      # kept small so free Colab doesn't run out of RAM
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    gradient_accumulation_steps=4,   # makes up for the smaller batch size above
    gradient_checkpointing=True,     # uses less memory, trains a little slower
    save_strategy="epoch",
    save_total_limit=1,
    fp16=use_fp16,      # mixed precision -- faster on free T4 GPU, skipped if no GPU
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer  # newer transformers renamed "tokenizer" to "processing_class"
)

trainer.train()

GPU available: True


Epoch,Training Loss,Validation Loss
1,2.706817,1.590722
2,1.425723,1.487169
3,1.124575,1.502702


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=1.6021864095052083, metrics={'train_runtime': 508.7142, 'train_samples_per_second': 58.972, 'train_steps_per_second': 3.686, 'total_flos': 2939694750720000.0, 'train_loss': 1.6021864095052083, 'epoch': 3.0})

### SECTION 6: Evaluate Accuracy

In [7]:
import collections

def get_predicted_answer(context, question):
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=384)
    with torch.no_grad():
        outputs = model(**{k: v.to(model.device) for k, v in inputs.items()})
    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits)
    answer_tokens = inputs["input_ids"][0][start_idx:end_idx + 1]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True)

# simple exact-match accuracy check on a sample of validation set
correct = 0
sample_n = 200
for i in range(sample_n):
    example = val_data[i]
    pred = get_predicted_answer(example["context"], example["question"])
    true_answers = example["answers"]["text"]
    if any(pred.strip().lower() == t.strip().lower() for t in true_answers):
        correct += 1

accuracy = correct / sample_n
print(f"Approx. exact-match accuracy on {sample_n} validation samples: {accuracy*100:.2f}%")

Approx. exact-match accuracy on 200 validation samples: 56.00%


### SECTION 7: Save Model for Deployment

In [8]:
model.save_pretrained("./domain-qa-bot")
tokenizer.save_pretrained("./domain-qa-bot")
print("Model saved to ./domain-qa-bot")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./domain-qa-bot
